# Experimental Design, A/B Testing, ANOVA, And Residuals

**Official MA1001B Alignment:** *7.1 experimental strategies; 7.2 ANOVA; 7.3 fixed effects; 7.4 residual analysis.*


## How To Use This Lesson

This notebook is designed as a guided teaching episode and interactive lab, not a passive code demonstration. To get the most out of this lesson:
1. **Read the conceptual explanations and explicit links** before running any code.
2. **Execute code cells sequentially**, paying attention to inline educational comments.
3. **Pause at the Guided Checkpoint** to discuss with a partner and write your reasoning before checking solutions.
4. **Complete the Independent Practice and Exit Ticket**; written justification is the primary evidence of statistical competence.


## Learning Goals

By the end of this lesson, you will be able to:
- Explain the foundational role of random assignment in protecting experimental comparisons from confounding variables.
- Execute One-Way Analysis of Variance (ANOVA) using SciPy (`stats.f_oneway`) to compare across three or more group means.
- Calculate experimental group means and isolate within-group residual variation (`response - group_mean`).
- Inspect residual distributions using histograms and boxplots to verify ANOVA model assumptions.


## The Three Explicit Links

In accordance with the MA1001B pedagogical framework, this lesson explicitly connects theory, computation, and action:

- **1. Conceptual Link (What is modeled):** We partition total data variation into between-group treatment effects and within-group residual noise.
- **2. Computational Link (How Python represents it):** We use Pandas group transformations (`.transform('mean')`) to compute residuals and SciPy (`stats.f_oneway`) for F-tests.
- **3. Decision Link (How it guides action):** ANOVA prevents inflating Type I error rates when comparing multiple interface designs, ensuring reliable design selection.


## Decision Scenario

> **The Problem:** A team compares three interface designs. The decision should use group differences, residual variation, and the quality of the experimental design.


## Conceptual Explanation

Experimental design is about creating credible comparisons. Random assignment protects against systematic differences between treatment groups. ANOVA compares between-group variability with within-group variability. Residual analysis checks whether the model leaves structure unexplained.


## Mathematical Anchor

ANOVA uses an F statistic: variation explained by groups divided by residual variation, adjusted by degrees of freedom.


## Data And Workflow Notes

Uses a simulated single-factor experiment with three designs.


## Practical Python Workflow

The following worked example demonstrates how to implement these statistical concepts in Python to generate evidence for decision making.


### Step 1: Experimental Data Simulation across 3 UI Designs

We simulate a single-factor experiment where n=210 users are randomly assigned across three interface designs (Design A, B, C) with different underlying response scores.


In [ ]:
# Import required data science and statistical libraries
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

# Set reproducible random seed and visual styling
rng = np.random.default_rng(1001)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 20)

# Simulate randomized experiment across 3 UI designs (n=70 per group)
experiment = pd.DataFrame({
    "design": np.repeat(["Design_A", "Design_B", "Design_C"], 70),
    "response_score": np.r_[rng.normal(50, 7, 70), rng.normal(54, 7, 70), rng.normal(58, 7, 70)],
})

# Summarize sample size, mean, and standard deviation by design
experiment.groupby("design")["response_score"].agg(
    users="count", mean_score="mean", std_dev="std"
).round(2)


### Step 2: One-Way ANOVA F-Test Execution

We extract the response score arrays for each design group and execute a One-Way ANOVA F-test to test the null hypothesis of equal group means.


In [ ]:
# Extract group response arrays and execute One-Way ANOVA
group_arrays = [group["response_score"].to_numpy() for _, group in experiment.groupby("design")]
f_stat, p_value = stats.f_oneway(*group_arrays)

pd.Series({
    "anova_F_statistic": f_stat,
    "p_value": p_value,
    "null_hypothesis_equal_means": "REJECTED (Significant group differences)" if p_value < 0.05 else "NOT REJECTED"
}).round(4)


### Step 3: Decomposing Effects & Computing Residuals

We calculate group-specific mean responses and subtract them from individual user scores to isolate unexplained within-group residuals (`residual = response - group_mean`).


In [ ]:
# Calculate group means and individual residual errors
group_means = experiment.groupby("design")["response_score"].transform("mean")
experiment["residual"] = experiment["response_score"] - group_means

# Display first 2 rows of each design group to verify residual decomposition
experiment.groupby("design").head(2).round(2)


### Step 4: Visualizing Treatment Effects & Residual Health

We create side-by-side plots: a boxplot comparing response scores across UI designs, and a histogram verifying that residuals follow a symmetric Normal distribution centered at zero.


In [ ]:
# Plot treatment effects and check residual normality assumption
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.boxplot(data=experiment, x="design", y="response_score", palette="Blues", ax=axes[0], width=0.4)
axes[0].set_title("Response Score by UI Design", fontsize=13)
axes[0].set_xlabel("Interface Design")
axes[0].set_ylabel("User Response Score")

sns.histplot(experiment["residual"], kde=True, ax=axes[1], color="purple", bins=25)
axes[1].axvline(0, color="black", linestyle="--", linewidth=1.5)
axes[1].set_title("Residual Error Distribution (Check Normality)", fontsize=13)
axes[1].set_xlabel("Residual (Observed - Group Mean)")
plt.tight_layout()
plt.show()


## Guided Checkpoint

> [!IMPORTANT]
> **Pair Discussion & Writing Prompt:**
> What specifically does a statistically significant ANOVA F-test tell us, and what critical information does it NOT tell us?

*Write your reasoned response below before continuing:*


## Common Mistakes & Statistical Pitfalls

Avoid these frequent errors when conducting or communicating this analysis:
- **Warning:** Treating observational group differences as causal experimental effects without verifying random assignment.
- **Warning:** Stopping at the omnibus ANOVA p-value without conducting pairwise follow-up comparisons or reporting effect sizes.
- **Warning:** Ignoring residual error patterns, severe outliers, or unequal within-group variances.


## Independent Practice

> [!TIP]
> **Your Task:**
> Estimate the pairwise mean differences between designs (`Mean_B - Mean_A`, `Mean_C - Mean_A`, `Mean_C - Mean_B`). Based on these differences and the residual spread, decide which UI design you recommend for adoption.

*Use the empty code and markdown cells below to implement your analysis and justify your recommendation.*


In [ ]:
# Write your independent practice code here
# Remember to inspect your outputs and check assumptions


## Decision Interpretation Template

Use this structured format to write your defensible conclusion and recommendation:

1. **The Decision Question:** *State the practical question being answered...*
2. **The Statistical Evidence:** *Summarize key metrics, intervals, p-values, or model comparisons...*
3. **Uncertainty & Limitations:** *Identify what the data cannot prove and what assumptions were made...*
4. **Actionable Recommendation:** *Therefore, I recommend [action] because [justification]...*


## Exit Ticket

> **Reflection:** Why is random assignment central to establishing credible causal evidence in experimental design?

*Write your brief conceptual reflection below:*
